# Δημιουργία Εφαρμογών Παραγωγής Εικόνας (OpenAI)

Τα LLM δεν περιορίζονται μόνο στην παραγωγή κειμένου. Μπορείτε επίσης να δημιουργήσετε εικόνες από περιγραφές κειμένου. Οι εικόνες ως μέσο είναι χρήσιμες στη MedTech, την αρχιτεκτονική, τον τουρισμό, την ανάπτυξη παιχνιδιών, το μάρκετινγκ και άλλα. Σε αυτό το μάθημα δουλεύουμε με τα σημερινά μοντέλα **GPT Image** στην πλατφόρμα OpenAI.

## Στόχοι μάθησης

Στο τέλος του μαθήματος θα μπορείτε να:

- Εξηγήσετε τι είναι η παραγωγή εικόνας και πού είναι χρήσιμη.
- Κατανοήσετε την οικογένεια μοντέλων `gpt-image` και πώς διαφέρει από τα παλαιότερα μοντέλα DALL·E.
- Δημιουργήσετε μια εφαρμογή παραγωγής εικόνας και να επεξεργαστείτε εικόνες.

## Τι είναι η παραγωγή εικόνας;

Τα μοντέλα παραγωγής εικόνας δημιουργούν εικόνες από ένα κείμενο-προτροπή. Σύγχρονα μοντέλα όπως το `gpt-image` μαθαίνουν τη σχέση ανάμεσα σε κείμενο και εικόνες κατά την εκπαίδευση, και στη συνέχεια μετατρέπουν επαναληπτικά τυχαίο θόρυβο σε μια εικόνα που ταιριάζει στην προτροπή σας.

Δύο γνωστές οικογένειες μοντέλων εικόνας είναι:

- **`gpt-image` (OpenAI)** — η τρέχουσα γενιά που χρησιμοποιείται σε αυτό το μάθημα. Υποστηρίζει παραγωγή εικόνας από κείμενο και επεξεργασία εικόνας (inpainting με μάσκα).
- **Midjourney** — ένα δημοφιλές μοντέλο τρίτου μέρους με δική του υπηρεσία και διαδικασία μέσω Discord.

> Τα παλαιότερα μοντέλα εικόνας της OpenAI — **DALL·E 2** και **DALL·E 3** — είναι παρωχημένα. Το DALL·E 3 δεν είναι πλέον διαθέσιμο για νέες υλοποιήσεις, και λειτουργίες όπως το `create_variation` υπήρχαν μόνο στο DALL·E 2. Χρησιμοποιήστε τα μοντέλα `gpt-image` για νέες εφαρμογές.

> **Σημαντικό:** Τα μοντέλα `gpt-image` επιστρέφουν την παραγόμενη εικόνα ως **base64** (`b64_json`), όχι ως URL. Ο κώδικάς σας αποκωδικοποιεί το string base64 σε bytes και το αποθηκεύει — δεν υπάρχει URL εικόνας για λήψη.


## Δημιουργία της πρώτης σας εφαρμογής γεννήτριας εικόνας

Τι χρειάζεται λοιπόν για να φτιάξετε μια εφαρμογή δημιουργίας εικόνων; Χρειάζεστε τις ακόλουθες βιβλιοθήκες:

- **python-dotenv**, συνιστάται ιδιαίτερα να χρησιμοποιήσετε αυτή τη βιβλιοθήκη για να κρατήσετε τα μυστικά σας σε ένα αρχείο *.env* μακριά από τον κώδικα.
- **openai**, αυτή η βιβλιοθήκη είναι αυτή που θα χρησιμοποιήσετε για να αλληλεπιδράσετε με το OpenAI API.
- **pillow**, για να δουλέψετε με εικόνες στην Python.


1. Δημιουργήστε ένα αρχείο *.env* με το παρακάτω περιεχόμενο:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Συγκεντρώστε τις παραπάνω βιβλιοθήκες σε ένα αρχείο που ονομάζεται *requirements.txt* ως εξής:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Στη συνέχεια, δημιουργήστε ένα εικονικό περιβάλλον και εγκαταστήστε τις βιβλιοθήκες:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Για τα Windows, χρησιμοποιήστε τις ακόλουθες εντολές για να δημιουργήσετε και να ενεργοποιήσετε το εικονικό σας περιβάλλον:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Προσθέστε τον ακόλουθο κώδικα σε αρχείο που ονομάζεται *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # εισαγάγετε dotenv
    dotenv.load_dotenv()

    # Δημιουργήστε το αντικείμενο OpenAI (διαβάζει το OPENAI_API_KEY από το .env σας)
    client = openai.OpenAI()


    try:
        # Δημιουργήστε μια εικόνα χρησιμοποιώντας το API δημιουργίας εικόνας
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Εισάγετε το κείμενο του προτροπής σας εδώ
            size='1024x1024',
            n=1
        )
        # Ορίστε τον κατάλογο για την αποθηκευμένη εικόνα
        image_dir = os.path.join(os.curdir, 'images')

        # Αν ο κατάλογος δεν υπάρχει, δημιουργήστε τον
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Αρχικοποιήστε τη διαδρομή της εικόνας (σημειώστε ότι ο τύπος αρχείου πρέπει να είναι png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Τα μοντέλα gpt-image επιστρέφουν την εικόνα ως base64 (b64_json), όχι ως URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Εμφανίστε την εικόνα στον προεπιλεγμένο προβολέα εικόνων
        image = Image.open(image_path)
        image.show()

    # πιάστε εξαιρέσεις
    except openai.BadRequestError as err:
        print(err)

    ```

Ας εξηγήσουμε αυτόν τον κώδικα:

- Πρώτα, εισάγουμε τις βιβλιοθήκες που χρειαζόμαστε, συμπεριλαμβανομένης της βιβλιοθήκης OpenAI, της βιβλιοθήκης dotenv, του module base64 και της βιβλιοθήκης Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Μετά από αυτό, δημιουργούμε τον πελάτη, ο οποίος διαβάζει το API key από το αρχείο ``.env``.

    ```python
    # Δημιουργήστε αντικείμενο OpenAI
    client = openai.OpenAI()
    ```

- Στη συνέχεια, δημιουργούμε την εικόνα:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Εισάγετε το κείμενο του προτροπής σας εδώ
        size='1024x1024',
        n=1
    )
    ```

    Τα μοντέλα `gpt-image` επιστρέφουν την εικόνα ως συμβολοσειρά **base64** στο `data[0].b64_json`. Την αποκωδικοποιούμε σε bytes και την γράφουμε σε αρχείο — δεν υπάρχει URL για λήψη.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Τέλος, ανοίγουμε την εικόνα και χρησιμοποιούμε τον τυπικό προβολέα εικόνων για να την εμφανίσουμε:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Περισσότερες λεπτομέρειες για τη δημιουργία εικόνας

Ας δούμε τις παραμέτρους της `images.generate`:

- **model**, είναι το μοντέλο εικόνας που χρησιμοποιείται (για παράδειγμα `gpt-image-1`).
- **prompt**, είναι η κειμενική περιγραφή που χρησιμοποιείται για τη δημιουργία της εικόνας. Εδώ είναι "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, είναι το μέγεθος της δημιουργημένης εικόνας (για παράδειγμα `1024x1024`, `1536x1024`, `1024x1536`, ή `"auto"`).
- **n**, είναι ο αριθμός των παραγόμενων εικόνων. Εδώ δημιουργούμε μία.

> Τα μοντέλα εικόνας δεν παίρνουν παράμετρο `temperature` — αυτό είναι έλεγχος για τη δημιουργία κειμένου. Για ποικιλία, καλέστε ξανά το API· για να μειώσετε την ποικιλία, κάντε το prompt σας πιο συγκεκριμένο.

## Πρόσθετες δυνατότητες δημιουργίας εικόνας

Έχετε δει πώς να δημιουργήσετε μια εικόνα με λίγες γραμμές Python. Τα μοντέλα `gpt-image` μπορούν επίσης να **επεξεργαστούν** μια υπάρχουσα εικόνα. Παρέχοντας μια εικόνα, μια προαιρετική **μάσκα** (που σηματοδοτεί την περιοχή για αλλαγή), και ένα prompt, μπορείτε να αλλάξετε μέρος μιας εικόνας — για παράδειγμα, να προσθέσετε ένα καπέλο στο κουνελάκι μας.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# οι επεξεργασίες επιστρέφονται επίσης ως base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Η βασική εικόνα περιέχει μόνο το κουνελάκι· η τελική εικόνα έχει το καπέλο στο κουνελάκι.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
